# N3 — Collections Risk

## Decision question

Does the payment-timing model improve on a simple rule, and which receivables
deserve scarce collections attention?

The model is tested on a chronological holdout. A more complex model is useful
only if it improves a simple industry-median benchmark.


In [ ]:
from pathlib import Path
import json
import sys

# Find the public package locally. A fresh Colab runtime downloads the same
# participant-safe assets from the repository.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    for source_candidate in (candidate / 'src', candidate / 'CFOPackV002' / 'src'):
        if (source_candidate / 'workshop_bootstrap.py').exists():
            sys.path.insert(0, str(source_candidate))
            break

try:
    from workshop_bootstrap import bootstrap
except ImportError:
    from urllib.request import urlopen
    bootstrap_url = (
        'https://raw.githubusercontent.com/VinayaSharada/'
        'KateelLearningDemosToStudents/cfopack-v002-v2.1.0-beta.1/CFOPackV002/src/workshop_bootstrap.py'
    )
    namespace = {}
    exec(compile(urlopen(bootstrap_url).read(), bootstrap_url, 'exec'), namespace)
    bootstrap = namespace['bootstrap']

ROOT, OUTPUT_DIR = bootstrap()
from cfopack_v002 import (
    analyze_fx,
    default_decisions,
    load_inputs,
    load_manifest,
    reveal_team_shock,
    run_pipeline,
)
import workshop_visuals as viz
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    # Keep the notebooks runnable from a minimal local Python environment as
    # well as Colab/Jupyter. Rich notebook rendering remains the default.
    def Markdown(value):
        return value

    def display(value):
        print(value)

manifest = load_manifest(ROOT / 'config' / 'scenario_manifest.json')
decision_file = OUTPUT_DIR / 'N0_team_decisions.json'
if decision_file.exists():
    DECISIONS = json.loads(decision_file.read_text(encoding='utf-8'))
else:
    DECISIONS = default_decisions(manifest)


In [ ]:
data = load_inputs(ROOT / 'data' / 'synthetic')
viz.data_snapshot(data, OUTPUT_DIR, 'N3')


In [ ]:
summary = run_pipeline(ROOT, OUTPUT_DIR, DECISIONS)
print(f"Scenario {summary['scenario_version']} calculated for {DECISIONS['team_name']} (model cache: {'hit' if summary['model_cache_hit'] else 'rebuilt'})")


## Model value and limitations


In [ ]:
model_card = pd.read_csv(OUTPUT_DIR / 'N3_model_card.csv')
importance = pd.read_csv(OUTPUT_DIR / 'N3_feature_importance.csv')
predictions = pd.read_csv(OUTPUT_DIR / 'N3_collection_predictions.csv')
display(model_card)
display(importance)
viz.model_chart(model_card, importance, predictions, OUTPUT_DIR)


## Prioritized collections view


In [ ]:
priority = predictions.head(20)[[
    'invoice_id', 'customer_id', 'amount_usd', 'segment', 'key_account',
    'predicted_days_late', 'p75_days_late', 'prediction_spread_days'
]]
display(priority)
print(f"Top 20 exposure: ${priority['amount_usd'].sum():,.0f}")
print('High uncertainty is a reason for human review, not automatic escalation.')


## Team decision

Would you approve the model for prioritization, forecasting, both, or neither?
State the benchmark, error level, uncertainty, and human control supporting your
choice.


In [ ]:
# This choice changes N4 and every downstream receipt forecast.
DECISIONS['model_use'] = 'model'  # model or baseline
decision_file.write_text(json.dumps(DECISIONS, indent=2), encoding='utf-8')
summary = run_pipeline(ROOT, OUTPUT_DIR, DECISIONS)
print(f"Forecast method approved: {DECISIONS['model_use']}")


### Before moving on

Record your interpretation in the participant workbook. Do not copy a chart
without also recording the assumption and decision it supports.
